In [1]:
import os
from datasets import load_dataset
from dotenv import load_dotenv
from openai import OpenAI
from transformers import AutoTokenizer

# Load environment variables
load_dotenv()

# Initialize Groq client
client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ["GROQ_API_KEY"],
)

MODEL_NAME = "llama-3.3-70b-versatile"


In [2]:
ds = load_dataset("mrSoul7766/ECTSum")
transcript = ds["train"][1]["text"]
print(transcript[:1983])

I'm joined by Tom Greco, our President and Chief Executive Officer; and Jeff Shepherd, our Executive Vice President and Chief Financial Officer.
We also hope that you and your families are healthy and safe.
The health and safety of our team members and customers has been a top priority over the past year.
With strength across all channels, we delivered comparable store sales growth of 24.7%, and margin expansion of 478 basis points versus the prior year.
On a two-year stack, our comp sales growth was 15.4%.
Adjusted diluted earnings per share of $3.34 represented an all-time quarterly high for AAP, and improved more than 230% compared to Q1 2020.
Free cash flow of $259 million was up significantly versus the prior year, and we returned over $203 million to our shareholders through a combination of share repurchases and our quarterly cash dividend.
In addition, we recently announced an updated capital allocation framework targeting top quartile total shareholder return, highlighted by o

In [3]:
SUMMARIZE_PROMPT = """Generate a concise summary of the information in the following earnings call transcript.

Only respond with the summary, do not include any extraneous text.

Transcript:

{transcript}
"""

def summarize(transcript, n=1):
    prompt = SUMMARIZE_PROMPT.format(
        transcript=transcript
    )
    messages = [
        {
            "role": "user",
            "content": prompt
        },
    ]
    summaries = []
    for _ in range(n):
        resp = client.chat.completions.create(
            model=MODEL_NAME,
            messages=messages,
            temperature=0.9,
        )
        summaries.append(
            resp.choices[0].message.content
        )
    return summaries

In [4]:
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "user", "content": transcript}
    ]
)

summary = response.choices[0].message.content
print(summary)

This is a transcript of the quarterly earnings call of Advance Auto Parts (AAP), a leading automotive aftermarket parts provider. The call is hosted by Tom Greco, President and CEO, and Jeff Shepherd, EVP and CFO. Here are the key points discussed during the call:

**Financial Highlights**

* Net sales increased 23.4% to $3.3 billion
* Adjusted gross profit margin expanded 91 basis points to 44.8%
* Adjusted SG&A expense was $1.2 billion, representing 35.8% of net sales
* Adjusted operating income increased to $299 million, with a margin expansion of 478 basis points to 9%
* Adjusted diluted earnings per share was $3.34, up from $1.00 a year ago
* Free cash flow was $259 million, an increase of $330 million compared to last year

**Business Update**

* The company delivered comparable store sales growth of 24.7% and margin expansion of 478 basis points versus the prior year
* The company is building an ownership culture and a differentiated operating model
* The federal stimulus packag

In [5]:
JUDGE_PROMPT_V1 = """
Rate the following summary of an earnings call transcript on a 
scale from 1 to 10. 

1 means the summary is very poor, 10 means the summary is very good.

Provide reasoning followed by the final score at the end 
surrounded by <score> tags.

For example:

<score>1</score>

Transcript:

{transcript}

Summary:

{summary}
"""

def judge_reward_v1(
    transcript: str,
    summary: str, 
    model: str = "llama-3.3-70b-versatile", 
    verbose: bool = False,
) -> float:

    prompt = JUDGE_PROMPT_V1.format(
        transcript=transcript, 
        summary=summary,
    )

    messages = [
        {
            "role": "user",
            "content": prompt
        },
    ]

    resp = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0,
        max_tokens=300,
    )

    completion = resp.choices[0].message.content

    if verbose:
        print(completion)

    try:

        match = re.search(
            r"<score>\s*(\d+)\s*</score>",
            completion,
            re.IGNORECASE
        )

        if match is None:
            return 0.0

        score = int(match.group(1).strip())

    except:
        score = 0

    return score / 10

In [6]:
import re
score = judge_reward_v1(transcript, summary, verbose=True)
print(score)

The provided summary is a comprehensive and well-structured overview of the earnings call transcript of Advance Auto Parts (AAP). It effectively captures the key points discussed during the call, including financial highlights, business updates, and guidance.

The summary is well-organized, with clear headings and concise bullet points that make it easy to follow. It covers all the essential topics, such as financial performance, business trends, and strategic initiatives. The language used is straightforward, and the summary avoids using technical jargon or complex financial terminology that might be difficult for non-experts to understand.

One of the strengths of the summary is its ability to distill complex information into easily digestible bits. For example, it breaks down the company's financial performance into key metrics, such as net sales, gross profit margin, and adjusted operating income. This makes it easy for readers to quickly grasp the company's financial situation and

In [ ]:
resp = summarize(transcript, n=8)

summaries = resp

for s in summaries:
    print(s)